In [1]:
## Vloco de codigo para instalar versao especifica dos pacotes
##pip install pandas==1.5.3
##pip install numpy==1.24.3

In [2]:
import sys
import os

# Acesso aos módulos do diretório
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
print("Project root:", project_root)

Project root: C:\pod\hackathon_pod_2025


##### Carregando pacotes

In [3]:
# Pacotes de manipulacao

import pandas as pd
import numpy as np
import os

# Pacotes de visualizacao
import matplotlib.pyplot as plt
import seaborn as sns

# Funcoes customizadas
import configs.function_basic as funcoes

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 1.5.3
numpy: 1.24.3


## Carregando databases

#### Book_01

In [4]:
# Carregando book_01
book_01 = pd.read_csv(project_root/'database/processed/book_variaveis_01.csv', sep=',')
print("Book 01 data shape:", book_01.shape)

Book 01 data shape: (1290526, 12)


In [5]:
book_01.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290526 entries, 0 to 1290525
Data columns (total 12 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   SAFRA            1290526 non-null  int64  
 1   FLAG_INSTALACAO  1290526 non-null  int64  
 2   FPD              1290526 non-null  int64  
 3   PROD             1290526 non-null  object 
 4   flag_mig2        1290526 non-null  object 
 5   SCORE_01         1281087 non-null  float64
 6   SCORE_02         1289950 non-null  float64
 7   NUM_CPF          1290526 non-null  object 
 8   SCORE_RATEO      1280511 non-null  float64
 9   SCORE_AVG        1280511 non-null  float64
 10  SCORE_DIFF       1280511 non-null  float64
 11  SCORE_MIN        1290526 non-null  float64
dtypes: float64(6), int64(3), object(3)
memory usage: 118.2+ MB


#### Base Dados Cadastrais

In [6]:
## Carregando todos arquivos em parquet de uma pasta

all_files = [os.path.join(project_root/'database/raw/base_dados_cadastrais/', f) for f in os.listdir(project_root/'database/raw/base_dados_cadastrais/') if f.endswith('.parquet')]
df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_files]

df_dados_cadastrais = pd.concat(df_list, ignore_index=True)
print('Base Dados Cadastrais data shape:', df_dados_cadastrais.shape)

Base Dados Cadastrais data shape: (3900378, 33)


In [7]:
df_dados_cadastrais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900378 entries, 0 to 3900377
Data columns (total 33 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   NUM_CPF           object
 1   SAFRA             object
 2   FLAG_INSTALACAO   object
 3   FPD               object
 4   PROD              object
 5   flag_mig2         object
 6   STATUSRF          object
 7   DATADENASCIMENTO  object
 8   var_03            object
 9   var_02            object
 10  var_04            object
 11  var_05            object
 12  var_06            object
 13  var_07            object
 14  var_08            object
 15  var_09            object
 16  var_10            object
 17  var_11            object
 18  var_12            object
 19  var_13            object
 20  var_14            object
 21  var_15            object
 22  var_16            object
 23  var_17            object
 24  var_18            object
 25  var_19            object
 26  var_20            object
 27  var_21      

#### Merge dos Datasets

In [8]:
# Primeiro vamos transformar a coluna SAFRA para o mesmo formato de book_01
df_dados_cadastrais['SAFRA'] = df_dados_cadastrais['SAFRA'].astype('int64')

In [9]:
cols_to_drop = [
    col for col in df_dados_cadastrais.columns
    if col in book_01.columns
    and col not in ['SAFRA', 'NUM_CPF']
]

df_dados_cadastrais_clean = df_dados_cadastrais.drop(columns=cols_to_drop)

df_book_02 = pd.merge(
    book_01,
    df_dados_cadastrais_clean,
    how='left',
    on=['SAFRA', 'NUM_CPF']
)

In [10]:
# Sanity check
book_01.shape[0] == df_book_02.shape[0]

True

### Feature Engineer


##### Bloco 01
Primeiro iremos aplicar duas regras de negócio:
- Utilizaremos apenas registros que possuam `STATUSRF` = *REGULAR*
- Apenas CPFs que possuam a partir de 18 anos na data de contratação do plano
    - `SAFRA` - `DATADENASCIMENTO` >= 18

##### Bloco 02
Após a remoção dos valores acima, o próximo sanity check foi limpar colunas com alto indice de nulos e cardinalidade igual a 1

##### Bloco 03
Ajustes de colunas:

- Criar regiões a partir de `CEP_3_digitos` : 
    - 0–1 → Sudeste
    - 2–3 → Sudeste
    - 4 → Sudeste
    - 5 → Nordeste
    - 6–7 → Centro-Oeste / Norte
    - 8–9 → Sul

- As colunas a seguir são flags, possuem apenas um valor possível, sendo ele preenchido para casos positivos e null em casos negativos
    - `var_19` e `var_21` : estar iremos transformar em 0 e 1 e mudar o nome da coluna
    - A coluna `var_25` possui a concatenação das duas colunas acima


#### Bloco 01

In [11]:
# Filtrando CPFs que possuam STATUSRF = REGULAR
df_book_02 = df_book_02[df_book_02['STATUSRF'] == 'REGULAR']

In [12]:
df_book_02['STATUSRF'].value_counts()

REGULAR    1281602
Name: STATUSRF, dtype: int64

In [13]:
# Transformando coluna DATADENASCIMENTO em datetime
df_book_02['DATADENASCIMENTO'] = pd.to_datetime(df_book_02['DATADENASCIMENTO'], format='%d/%m/%Y')

# Criando uma coluna DATA_SAFRA a partir da coluna SAFRA
df_book_02['DATA_SAFRA'] = pd.to_datetime(df_book_02['SAFRA'].astype(str), format='%Y%m')

# Calculando a idade a partir da data de contratação da SAFRA
df_book_02 = funcoes.calcular_idade_df(df_book_02, 'DATADENASCIMENTO', 'DATA_SAFRA', 'IDADE')

# Selecionando apenas CPFs com idade maior que 18 anos
df_book_02 = df_book_02[df_book_02['IDADE'] >= 18]

In [14]:
df_book_02['IDADE'].describe()

count    1280828.0
mean     42.398951
std      14.286421
min           18.0
25%           31.0
50%           41.0
75%           52.0
max          125.0
Name: IDADE, dtype: Float64

In [15]:
df_book_02

,SAFRA,FLAG_INSTALACAO,FPD,PROD,flag_mig2,SCORE_01,SCORE_02,NUM_CPF,SCORE_RATEO,SCORE_AVG,...,var_19,var_20,var_21,var_22,var_23,var_24,var_25,CEP_3_digitos,DATA_SAFRA,IDADE
0,202410,1,0,CMV,PRE,562.0,636.0,ZZZZZX7XWY8,1.131673,599.0,...,None,None,FUNC_PRIVADO,None,None,ADMITIDO,FUNC_PRIVADO,479,2024-10-01,40
1,202410,1,1,CMV,PRE,546.0,518.0,ZZZZZX88YXY,0.948718,532.0,...,None,None,FUNC_PRIVADO,None,None,ADMITIDO,FUNC_PRIVADO,690,2024-10-01,43
2,202410,1,0,CMV,PRE,621.0,750.0,ZZZZZYT7XYT,1.207729,685.5,...,None,FUNC_PUBL,FUNC_PRIVADO,None,None,ADMITIDO,FUNC_PUBL FUNC_PRIVADO,790,2024-10-01,42
3,202410,1,1,CMV,PRE,609.0,679.0,ZZZZZNTXY9Z,1.114943,644.0,...,AUX_EMRG,None,None,None,None,None,AUX_EMRG,691,2024-10-01,36
4,202410,1,0,CMV,PRE,621.0,722.0,ZZZZZ79ZXUX,1.162641,671.5,...,None,None,FUNC_PRIVADO,None,None,ADMITIDO,FUNC_PRIVADO,758,2024-10-01,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1290521,202503,1,0,CMV,PRE,604.0,674.0,99997YWXNZZ,1.115894,639.0,...,AUX_EMRG,None,None,None,None,None,AUX_EMRG,339,2025-03-01,65
1290522,202503,1,0,CMV,PRE,688.0,765.0,99998TYXZN8,1.111919,726.5,...,None,None,FUNC_PRIVADO,None,None,ADMITIDO,APOSENTADO FUNC_PRIVADO,882,2025-03-01,42
1290523,202503,1,0,CMV,PRE,616.0,630.0,9999888YYU9,1.022727,623.0,...,AUX_EMRG,None,FUNC_PRIVADO,EMPR/DIRETOR,None,ADMITIDO,AUX_EMRG FUNC_PRIVADO EMPR/DIRETOR,769,2025-03-01,36
1290524,202503,1,0,CMV,PRE,627.0,649.0,9999889ZN9X,1.035088,638.0,...,AUX_EMRG,None,FUNC_PRIVADO,None,BOLSA_FAMILIA,ADMITIDO,AUX_EMRG FUNC_PRIVADO BOLSA_FAMILIA,339,2025-03-01,61


#### Bloco 02

Nesta seção iremos lidar com valores ausentes e valores continuos
- Deleção de colunas que possuam o limiar de 70% de valores nulos
- Deleção de colunas com cardinalidade igual a 1

In [16]:
funcoes.generate_metadata(df_book_02)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,var_11,object,1267568,98.96,4949
1,var_10,object,1266755,98.90,1520
2,var_20,object,1264534,98.73,1
3,var_02,object,1209445,94.43,1663
4,var_14,object,1182688,92.34,23
5,var_22,object,1182688,92.34,1
6,var_13,object,1083791,84.62,1494
7,var_07,object,1066916,83.30,86723
8,var_23,object,1053206,82.23,1
9,var_16,object,1053206,82.23,1258


In [17]:
# Deletando colunas que possuam mais que 70% de valores faltantes
threshold = 0.7
df_book_02 = funcoes.drop_columns_high_missing(df_book_02, threshold)

🧹 Colunas removidas (> 70% missing): 15
 - var_02: 94.4%
 - var_06: 80.8%
 - var_07: 83.3%
 - var_08: 80.8%
 - var_10: 98.9%
 - var_11: 99.0%
 - var_13: 84.6%
 - var_14: 92.3%
 - var_15: 82.2%
 - var_16: 82.2%
 - var_17: 82.2%
 - var_18: 80.8%
 - var_20: 98.7%
 - var_22: 92.3%
 - var_23: 82.2%


In [18]:
# Agora iremos deletar colunas que possuem cardinalidade igual a 1
df_book_02 = funcoes.drop_single_cardinality_columns(df_book_02)

🧹 Colunas removidas (cardinalidade = 1): 4
 - FLAG_INSTALACAO
 - PROD
 - flag_mig2
 - STATUSRF


In [19]:
# Conferindo metadados
funcoes.generate_metadata(df_book_02)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,var_09,object,680708,53.15,15
1,var_19,object,680708,53.15,1
2,var_21,object,492911,38.48,1
3,var_24,object,492911,38.48,2
4,var_12,object,492911,38.48,12956
5,var_25,object,131241,10.25,61
6,var_03,object,82368,6.43,100
7,CEP_3_digitos,object,74605,5.82,901
8,var_05,object,52789,4.12,10
9,SCORE_RATEO,float64,8036,0.63,77594


#### Bloco 03

Quebra de informação da coluna `CEP_3_digitos` em:
 - `REGIAO_POSTAL`
 - `REGIAO_POSTAL_TXT`
 - `SUB_REGIAO_POSTAL`

 Transformação das colunas `var_19`, `var_21` em booleana:
 - `var_19` -> `AUX_EMRG`: 1 para quem teve essa coluna preenchida, 0 para quem não teve
 - `var_21` -> `FUNC_PRIVADO`: 1 para quem teve essa coluna preenchida, 0 para quem não teve
 
 Criação da coluna `TEMPO_CADASTRO`
 - Esta coluna só é preenchida quando a coluna `var_21`possui preenchimento
 - Iremos inferir que é quando o cadastro foi preenchido e calcularemos a nova coluna
 - Após a criação desta coluna foi realizado o drop da coluna `var_12`

In [20]:
# Criando colunas a partir de CEP_3_digitos
df_book_02 = funcoes.mapear_regiao_subregiao_texto(df_book_02, 'CEP_3_digitos')

In [21]:
# Verificando apenas as variaveis que iniciam com var_
df_book_02.filter(like='var_')

,var_03,var_04,var_05,var_09,var_12,var_19,var_21,var_24,var_25
0,33,0,2,None,15/04/2009,None,FUNC_PRIVADO,ADMITIDO,FUNC_PRIVADO
1,14,0,2,None,14/04/2020,None,FUNC_PRIVADO,ADMITIDO,FUNC_PRIVADO
2,33,0,4,None,03/11/2010,None,FUNC_PRIVADO,ADMITIDO,FUNC_PUBL FUNC_PRIVADO
3,50,0,1,9,None,AUX_EMRG,None,None,AUX_EMRG
4,33,0,2,None,12/09/2018,None,FUNC_PRIVADO,ADMITIDO,FUNC_PRIVADO
...,...,...,...,...,...,...,...,...,...
1290521,43,0,2,8,None,AUX_EMRG,None,None,AUX_EMRG
1290522,67,0,3,None,19/11/2018,None,FUNC_PRIVADO,ADMITIDO,APOSENTADO FUNC_PRIVADO
1290523,29,0,3,9,10/08/2019,AUX_EMRG,FUNC_PRIVADO,ADMITIDO,AUX_EMRG FUNC_PRIVADO EMPR/DIRETOR
1290524,None,0,2,9,16/02/2012,AUX_EMRG,FUNC_PRIVADO,ADMITIDO,AUX_EMRG FUNC_PRIVADO BOLSA_FAMILIA


In [22]:
# Criando print de values_counts() das variaveis que começam com var_
for col in df_book_02.filter(like='var_').columns:
    print(f"Value counts for column: {col}")
    print(df_book_02[col].value_counts())
    print("\n")

Value counts for column: var_03
33    390813
1      87543
3      55689
50     35642
17     34205
       ...  
68       264
72       232
66       186
54       166
74       139
Name: var_03, Length: 100, dtype: int64


Value counts for column: var_04
0    1168350
1      69034
2      26676
3      10000
4       3840
5       2928
Name: var_04, dtype: int64


Value counts for column: var_05
1     597070
2     465435
3      85181
4      43888
5      23054
6       5286
7       4143
9       1917
8       1899
10       166
Name: var_05, dtype: int64


Value counts for column: var_09
9     354993
8      81553
5      54496
7      48379
6      21311
4      15969
3       8794
2       7784
1       6756
10        64
11        10
17         5
13         2
14         2
16         2
Name: var_09, dtype: int64


Value counts for column: var_12
01/10/2019    2939
01/04/2019    2908
03/02/2020    2865
01/08/2019    2855
01/02/2019    2814
              ... 
27/11/2005       1
14/02/1975       1
28/02/1989   

As variaveis `var_03`, `var_04`, `var_05`, `var_09` representam valores numericos continuos.

Iremos analisar a coluna `var_12` possui dados do tipo datetime e esta preenchida apenas quando possui a coluna `var_24`, com os dois possuindo aproximadamente 38% de nulos.

Iremos criar uma nova coluna de tempo de admissao ou dispensa

In [23]:
# Conferindo se sempre que temos registro na coluna var_12 eu tambem possuo registro na coluna var_24
(df_book_02['var_12'].notna() & df_book_02['var_24'].isna()).any()

False

In [24]:
# Transformando a coluna var_12 em datetime
df_book_02['var_12'] = pd.to_datetime(df_book_02['var_12'], format='%d/%m/%Y')

# Criando tempo de cadastro
df_book_02 = funcoes.calcular_idade_df(df_book_02, 'var_12', 'DATA_SAFRA', 'TEMPO_CADASTRO')

In [25]:
df_book_02['TEMPO_CADASTRO'].describe()

count    787917.0
mean     9.793863
std      6.364046
min           3.0
25%           5.0
50%           8.0
75%          12.0
max          67.0
Name: TEMPO_CADASTRO, dtype: Float64

In [30]:
# Agora iremos realizar o drop da coluna var_12
df_book_02 = df_book_02.drop(columns=['var_12'])

Agora iremos transformar a colunas colunas que tem comportamento de preenchimento booelano
 - `var_19`
 - `var_21`

In [26]:
# Aplicando a funcao para transformar as colunas em booleanas
colunas_booleanas = ['var_19', 'var_21']

df_book_02 = funcoes.criar_flag_com_nome_do_valor(df_book_02,colunas_booleanas)


In [28]:
df_book_02

,SAFRA,FPD,SCORE_01,SCORE_02,NUM_CPF,SCORE_RATEO,SCORE_AVG,SCORE_DIFF,SCORE_MIN,DATADENASCIMENTO,...,var_25,CEP_3_digitos,DATA_SAFRA,IDADE,REGIAO_POSTAL,SUB_REGIAO_POSTAL,REGIAO_POSTAL_TXT,TEMPO_CADASTRO,AUX_EMRG,FUNC_PRIVADO
0,202410,0,562.0,636.0,ZZZZZX7XWY8,1.131673,599.0,74.0,562.0,1983-12-26,...,FUNC_PRIVADO,479,2024-10-01,40,4,47,Bahia e Sergipe,15,0,1
1,202410,1,546.0,518.0,ZZZZZX88YXY,0.948718,532.0,-28.0,518.0,1980-12-24,...,FUNC_PRIVADO,690,2024-10-01,43,6,69,"Nordeste Setentrional (CE, PI, MA)",4,0,1
2,202410,0,621.0,750.0,ZZZZZYT7XYT,1.207729,685.5,129.0,621.0,1982-07-23,...,FUNC_PUBL FUNC_PRIVADO,790,2024-10-01,42,7,79,Centro-Oeste e Norte,13,0,1
3,202410,1,609.0,679.0,ZZZZZNTXY9Z,1.114943,644.0,70.0,609.0,1988-09-27,...,AUX_EMRG,691,2024-10-01,36,6,69,"Nordeste Setentrional (CE, PI, MA)",<NA>,1,0
4,202410,0,621.0,722.0,ZZZZZ79ZXUX,1.162641,671.5,101.0,621.0,1984-08-04,...,FUNC_PRIVADO,758,2024-10-01,40,7,75,Centro-Oeste e Norte,6,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1290521,202503,0,604.0,674.0,99997YWXNZZ,1.115894,639.0,70.0,604.0,1960-01-05,...,AUX_EMRG,339,2025-03-01,65,3,33,Minas Gerais,<NA>,1,0
1290522,202503,0,688.0,765.0,99998TYXZN8,1.111919,726.5,77.0,688.0,1982-12-25,...,APOSENTADO FUNC_PRIVADO,882,2025-03-01,42,8,88,Paraná e Santa Catarina,6,0,1
1290523,202503,0,616.0,630.0,9999888YYU9,1.022727,623.0,14.0,616.0,1988-12-26,...,AUX_EMRG FUNC_PRIVADO EMPR/DIRETOR,769,2025-03-01,36,7,76,Centro-Oeste e Norte,5,1,1
1290524,202503,0,627.0,649.0,9999889ZN9X,1.035088,638.0,22.0,627.0,1963-09-15,...,AUX_EMRG FUNC_PRIVADO BOLSA_FAMILIA,339,2025-03-01,61,3,33,Minas Gerais,13,1,1


In [31]:
funcoes.generate_metadata(df_book_02)

,nome_variavel,tipo,qt_nulos,percent_nulos,cardinalidade
0,var_09,object,680708,53.15,15
1,TEMPO_CADASTRO,Int64,492911,38.48,62
2,var_24,object,492911,38.48,2
3,var_25,object,131241,10.25,61
4,var_03,object,82368,6.43,100
5,REGIAO_POSTAL_TXT,object,74605,5.82,10
6,CEP_3_digitos,object,74605,5.82,901
7,var_05,object,52789,4.12,10
8,SCORE_DIFF,float64,8036,0.63,1158
9,SCORE_AVG,float64,8036,0.63,1227


In [32]:
df_book_02

,SAFRA,FPD,SCORE_01,SCORE_02,NUM_CPF,SCORE_RATEO,SCORE_AVG,SCORE_DIFF,SCORE_MIN,DATADENASCIMENTO,...,var_25,CEP_3_digitos,DATA_SAFRA,IDADE,REGIAO_POSTAL,SUB_REGIAO_POSTAL,REGIAO_POSTAL_TXT,TEMPO_CADASTRO,AUX_EMRG,FUNC_PRIVADO
0,202410,0,562.0,636.0,ZZZZZX7XWY8,1.131673,599.0,74.0,562.0,1983-12-26,...,FUNC_PRIVADO,479,2024-10-01,40,4,47,Bahia e Sergipe,15,0,1
1,202410,1,546.0,518.0,ZZZZZX88YXY,0.948718,532.0,-28.0,518.0,1980-12-24,...,FUNC_PRIVADO,690,2024-10-01,43,6,69,"Nordeste Setentrional (CE, PI, MA)",4,0,1
2,202410,0,621.0,750.0,ZZZZZYT7XYT,1.207729,685.5,129.0,621.0,1982-07-23,...,FUNC_PUBL FUNC_PRIVADO,790,2024-10-01,42,7,79,Centro-Oeste e Norte,13,0,1
3,202410,1,609.0,679.0,ZZZZZNTXY9Z,1.114943,644.0,70.0,609.0,1988-09-27,...,AUX_EMRG,691,2024-10-01,36,6,69,"Nordeste Setentrional (CE, PI, MA)",<NA>,1,0
4,202410,0,621.0,722.0,ZZZZZ79ZXUX,1.162641,671.5,101.0,621.0,1984-08-04,...,FUNC_PRIVADO,758,2024-10-01,40,7,75,Centro-Oeste e Norte,6,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1290521,202503,0,604.0,674.0,99997YWXNZZ,1.115894,639.0,70.0,604.0,1960-01-05,...,AUX_EMRG,339,2025-03-01,65,3,33,Minas Gerais,<NA>,1,0
1290522,202503,0,688.0,765.0,99998TYXZN8,1.111919,726.5,77.0,688.0,1982-12-25,...,APOSENTADO FUNC_PRIVADO,882,2025-03-01,42,8,88,Paraná e Santa Catarina,6,0,1
1290523,202503,0,616.0,630.0,9999888YYU9,1.022727,623.0,14.0,616.0,1988-12-26,...,AUX_EMRG FUNC_PRIVADO EMPR/DIRETOR,769,2025-03-01,36,7,76,Centro-Oeste e Norte,5,1,1
1290524,202503,0,627.0,649.0,9999889ZN9X,1.035088,638.0,22.0,627.0,1963-09-15,...,AUX_EMRG FUNC_PRIVADO BOLSA_FAMILIA,339,2025-03-01,61,3,33,Minas Gerais,13,1,1


#### Ajustando os tipos de dados

Durante o processo inteiro os tipos de dados foram criadas da forma correta.

Apesar de serem valores de tipos numericos, as variaveis abaixo foram mantidas como objeto pelo tipo de dado inserido que se comportam como categóricos:
 - `var_02`
 - `var_04`
 - `var_05`
 - `var_09`

In [36]:
# Criando novo dataset
book_variaveis_02 = df_book_02.copy()

In [37]:
# Salvando o dataframe em csv
book_variaveis_02.to_csv(project_root/'database/processed/book_variaveis_02.csv', index=False)